In [ ]:
from fg.graph import Graph
from fg.variables import Variable, Parameter
from fg.gaussian import Gaussian
import torch
from torch import Tensor
import random
import copy
import matplotlib.pyplot as plt
import torch.nn.functional as F
import numpy as np
from collections import defaultdict

def h_dXdt(Xt, Yt, a, omega, X_ext = 0., G = 0.8):
    return (a - Xt**2 - Yt**2) * Xt - omega * Yt + G*X_ext + np.random.normal(0, 1)

def h_dYdt(Xt, Yt, a, omega, Y_ext = 0., G = 0.8):
    return (a - Xt**2 - Yt**2) * Yt + omega * Xt + G*Y_ext + np.random.normal(0, 1)


def simulate_SL(T, dt, a, omega, beta):
    np.random.seed(42)
    Ts = np.arange(0, T + dt, dt)

    X = [0.0]
    Y = [0.0]

    for t in range(len(Ts) - 1):
        Xtp = X[t]+ dt*(h_dXdt(X[t], Y[t], a, omega, 0, 0))
        Ytp = Y[t]+ dt*(h_dYdt(X[t], Y[t], a, omega, 0, 0))

        X.append(Xtp)
        Y.append(Ytp)

    return np.stack(X), np.stack(Y)

class Variable:
    def __init__(self, id, belief : Gaussian, graph : Graph, num_vars : int = 1, connected_factors = []) -> None:
        self.id = id
        self.belief = belief
        self.num_vars = num_vars

        self.inbox = {}
        self.connected_factors = connected_factors

        self.implicit_var = 0.0

        self.graph = graph

    @property
    def mean(self) -> Tensor:
        return self.belief.mean

    @property
    def cov(self) -> Tensor:
        return self.belief.cov

    @property
    def eta(self) -> Tensor:
        return self.belief.eta

    @property
    def lmbda(self) -> Tensor:
        return self.belief.lmbda

    # This is so the linter doesn't complain
    def send_initial_messages(self) -> None: pass
    
    def update_belief(self) -> None:
        '''
        Consume the messages in the inbox to update belief
        '''
        self.belief = Gaussian.zeros_like(self.belief)

        for _, message in self.inbox.items():
            self.belief *= message

        # if not torch.is_nonzero(curr.lmbda): print('We are having a serious problem in the variable')

    def compute_and_send_messages(self) -> None:
        '''
        Equation 2.50, 2.51 in Ortiz (2023)
        '''
        self.update_belief()

        for fid in self.connected_factors:
            if fid == -1: continue

            # Message can be efficiently computed by calculating belief and then
            # dividing with the incoming message?
            msg = self.belief / self.inbox.get(fid, Gaussian.zeros_like(self.belief))

            self.graph.send_msg_to_factor(self.id, fid, msg)

    def __str__(self):
        return f'Variable {self.id} [n = {self.num_vars}, mu={self.mean}, cov={self.cov}]'


class ObservationFactor:
    def __init__(self, factor_id, var_id, z, lmbda_in, graph : Graph, huber = False) -> None:
        self.factor_id = factor_id
        self.var_id = var_id

        self.z = z

        self.lmbda_in = lmbda_in

        J = torch.eye(self.lmbda_in.shape[0])

        # Equation 2.46, 2.47 in Ortiz (2023)
        self.belief = Gaussian.from_canonical((J.T @ lmbda_in) @ z, (J.T @ lmbda_in) @ J)

        # Huber threshold
        self.N_sigma = torch.sqrt(self.lmbda_in[0,0])

        self.inbox = {}

        self.graph = graph

        self.huber = huber

    def update_belief(self) -> None: pass

    def compute_and_send_messages(self) -> None:
        kR = 1.

        message = self.belief * kR
        self.graph.send_msg_to_variable(self.factor_id, self.var_id, message)

    def __str__(self) -> str:
        return f'Obs: [{self.factor_id} -- {self.var_id}], z = {self.z}'


class PriorFactor:
    def __init__(self, factor_id, var_id, z, lmbda_in, graph : Graph, huber = False) -> None:
        self.factor_id = factor_id
        self.var_id = var_id

        self.z = z
        self.lmbda_in = lmbda_in

        self.J = torch.eye(self.lmbda_in.shape[0])

        # Equation 2.46, 2.47 in Ortiz (2023)
        self.belief = Gaussian.from_canonical((self.J.T @ lmbda_in) @ z, (self.J.T @ lmbda_in) @ self.J)

        # Huber threshold
        self.N_sigma = torch.sqrt(self.lmbda_in[0,0])

        self.inbox = {}

        self.graph = graph

    def update_belief(self) -> None:
        self.belief = Gaussian.from_canonical((self.J.T @ self.lmbda_in) @ self.z, (self.J.T @ self.lmbda_in) @ self.J)

    def compute_and_send_messages(self) -> None:
        kR = 1.

        message = self.belief * kR
        self.graph.send_msg_to_variable(self.factor_id, self.var_id, message)

    def __str__(self) -> str:
        return f'Prior: [{self.factor_id} -- {self.var_id}], z = {self.z}'


class DynamicsFactor:
    '''
    Represents a dynamics factor that enforces dynamics between `Et_id` (left) and `Etp_id` (right),
    and is also connected to learnable parameters given by `parameters`.
    '''
    def __init__(self, Xt_id, Xtp_id, lmbda_in : Tensor, factor_id, graph : Graph, huber = False, connected_params = [], dt = 0.01) -> None:
        self.Xt_id = Xt_id
        self.Xtp_id = Xtp_id
        self.dt = dt

        self.lmbda_in = lmbda_in
        self.factor_id = factor_id
        self.graph : Graph = graph

        self.parameters = connected_params

        self.N_sigma = torch.sqrt(lmbda_in)
        self.z = 0

        self.inbox = {}

        # Used for message damping, see Ortiz (2023) 3.4.6
        self._prev_messages = {}

        self._connected_vars = [Xt_id, Xtp_id] + list(self.parameters)

        self.huber = huber

    def h_fn(self, Xt, Xtp, a, omega):        
        Yt = self.graph.var_nodes[self.Xt_id].implicit_var
        Ytp = Yt + self.dt*h_dYdt(Xt, Yt, a, omega)
        self.graph.var_nodes[self.Xtp_id].implicit_var = Ytp.detach()
        
        h_X = Xtp - (Xt + self.dt*h_dXdt(Xt, Yt, a, omega))
        return h_X

    def linearise(self) -> Gaussian:
        '''
        Returns the linearised Gaussian factor based on equations 2.46 and 2.47 in Ortiz (2023)
        '''

        # Extracts the means of all the beliefs of our adj.
        # parameters and gets them ready for autograd
        connected_variables = []
        for i in self._connected_vars:
            mean = self.graph.get_var_belief(i).mean.detach().clone()
            connected_variables.append(mean.reshape(1, 1).requires_grad_(True))

        Xt_mu, Xtp_mu = connected_variables[0:2]
        a,omega = connected_variables[2:]

        self.h = self.h_fn(Xt_mu, Xtp_mu, a, omega)
        J = torch.concat(torch.autograd.functional.jacobian(self.h_fn, (Xt_mu, Xtp_mu, a, omega)), 0)[..., 0, 0].T
        x0 = torch.concat([v for v in connected_variables], dim=0)

        eta = (J.T @ self.lmbda_in) @ (-self.h.T + J @ x0)
        lmbda = (J.T @ self.lmbda_in) @ J

        return Gaussian.from_canonical(eta.detach(), lmbda.detach())

    def compute_huber(self) -> float:
        # Equation 3.16 in Ortiz (2023)
        r = self.z - self.h
        M = torch.sqrt(r @ self.lmbda_in @ r)

        # Equation 3.20 in Ortiz (2023)
        if M > self.N_sigma and self.huber:
            kR = (2 * self.N_sigma / M) - (self.N_sigma**2 / M**2)
            kR = kR.item()
        else:
            kR = 1.

        return kR

    def _compute_message_to_i(self, i, beta = 0.3) -> Gaussian:
        '''
        Compute message to variable at index i in `self._vars`,
        All of this is eqn 8 from 'Learning in Deep Factor Graphs with Gaussian Belief Propagation'
        '''
        linearised_factor = self.linearise()

        product = Gaussian.zeros_like(linearised_factor)

        # Build our message product by adding corresponding eta and lambda
        # in product
        k = 0
        for j, id in enumerate(self._connected_vars):
            if j != i:
                in_msg = self.inbox.get(id, Gaussian.from_canonical(torch.tensor([0.]), \
                    torch.tensor([0.])))

                # Element 0 and 1 in self._connected_vars will be the
                # EI oscillator vars, and they each have a 2D Gaussian as their belief
                # since they encode Et, It and Etp, Itp respectively. Therefore,
                # we have to correctly offset our product Gaussian with 2 if
                # our j is at the 0th or 1st element. Otherwise just continue as
                # normal.
                offset = in_msg.eta.numel()
                product.eta[k : k+offset] += in_msg.eta
                product.lmbda[k : k+offset, k : k+offset] += in_msg.lmbda

                k += offset
            else:
                k += self.graph.var_nodes[self._connected_vars[i]].num_vars

        factor_product = linearised_factor * product

        start_idx = 0
        for k in range(i):
            start_idx += self.graph.var_nodes[self._connected_vars[k]].num_vars

        idx_to_marginalise = list(range(start_idx, start_idx + self.graph.var_nodes[self._connected_vars[i]].num_vars))

        marginal = factor_product.marginalise(idx_to_marginalise)

        kR = 1.
        marginal *= kR

        prev_msg = self._prev_messages.get(i, Gaussian.zeros_like(marginal))
        damped_factor = (marginal * beta) * (prev_msg * (1 - beta))

        # Store previous message
        self._prev_messages[i] = damped_factor

        return damped_factor

    def compute_and_send_messages(self, damping = 0.7) -> None:
        for i, var_id in enumerate(self._connected_vars):

            if random.uniform(0,1) < damping:
                msg = self._compute_message_to_i(i)
                self.graph.send_msg_to_variable(self.factor_id, var_id, msg)

    def __str__(self):
        return f'Dynamics: [{self.Xt_id} -- {self.Xtp_id} -- {self.Yt_id} -- {self.Ytp_id}], z = {self.z}'


if __name__ == '__main__':
    sigma_obs = 1e-1
    sigma_dynamics = 5e-3
    sigma_prior = 5e1
    T = 1.5
    dt = 0.01
    iters = 100
    nr = 1

    results_mean = defaultdict(lambda: [])
    results_cov = defaultdict(lambda: [])
    results_E = defaultdict(lambda: [])
    results_I = defaultdict(lambda: [])

    # Create our ground truth signal
    gt_config = {
        'T': T,
        'dt': dt,
        'nr': nr,
        'a': torch.empty(1).uniform_(0.1, 10),
        'omega': torch.empty(1).uniform_(2*np.pi, 8*np.pi),
        'beta': torch.empty(1).uniform_(1.0, 1.0),
        'obs_noise': 0.0,
    }
    config = copy.deepcopy(gt_config)
    X_gt, _ = simulate_SL(T, dt, gt_config['a'][0], gt_config['omega'][0], gt_config['beta'][0])

    factor_graph = Graph(nr)
    time = torch.arange(0, len(X_gt), 1)

    param_list = ['a', 'omega']

    B = 16

    # ----------- Construct Factor Graph ----------- #
    # Create our X and Y variables separately as well as the obs. factor for X
    for t in range(len(time)):
        factor_graph.var_nodes[f'X_t{t}'] = Variable(
            id       = f'X_t{t}',
            belief   = Gaussian(torch.tensor([[0.1]]), torch.tensor([[0.2]])),
            graph    = factor_graph,
            num_vars = 1,
            connected_factors = [(f'osc_t{t}', f'osc_t{t+1}') if t+1 < len(time) else -1] +  [(f'osc_t{t-1}', f'osc_t{t}') if t > 0 else -1]
        )

        factor_graph.factor_nodes[f'obs_t{t}'] = ObservationFactor(
            factor_id = f'obs_t{t}',
            var_id    = f'X_t{t}',
            z         = torch.tensor([X_gt[t]]).float(),
            lmbda_in  = torch.tensor([[sigma_obs ** -2]]),
            graph     = factor_graph
        )

    # Create our learnable parameters
    for p in param_list:
        p_id = f'p({p})'
        factor_graph.param_ids.append(p_id)
        factor_graph.var_nodes[p_id] = Parameter(
            id     = p_id,
            belief = Gaussian(torch.tensor([[0.1]]), torch.tensor([[sigma_prior ** 2.]])),
            graph  = factor_graph,
            connected_factors = [(f'osc_t{t}', f'osc_t{t+1}') for t in range(len(time)-1)]
        )

        # Add priors to those parameters
        factor_graph.factor_nodes[f'{p_id}_prior'] = PriorFactor(
            factor_id = f'{p_id}_prior',
            var_id = p_id,
            z = torch.tensor([[1e-4]]),
            lmbda_in = torch.diag(torch.tensor([sigma_prior ** -2])),
            graph = factor_graph
        )

    # Create our dynamics factors
    for t in range(len(time)):
        if t+1 < len(time):
            dyn_id = (f'osc_t{t}', f'osc_t{t+1}')
            factor_graph.factor_nodes[dyn_id] = DynamicsFactor(
                Xt_id  = f'X_t{t}',
                Xtp_id = f'X_t{t+1}',
                lmbda_in = torch.tensor([[sigma_dynamics ** -2.]]),
                factor_id = dyn_id,
                graph = factor_graph,
                connected_params = [f'p({p})' for p in param_list],
                dt = dt
            )

    print('GT', gt_config)
    # Actually run GBP on our factor graph (sweep schedule)
    for iter in range(iters):
        if iter % 1 == 0:
            for a in param_list:
                results_mean[a].append(torch.abs((gt_config[a] - factor_graph.var_nodes[f'p({a})'].belief.mean.item()) / gt_config[a]))
                results_cov[a].append(factor_graph.var_nodes[f'p({a})'].belief.cov.item())

                if factor_graph.var_nodes[f'p({a})'].belief.eta.isnan().any(): 
                    print('Found nan, exiting..')
                    exit(0)
            
            # Recreate the signal with the learnt parameters and store the MAE
            for k in param_list:
                t = f'p({k})'
                config[k] = factor_graph.get_var_belief(t).mean

            config['obs_noise'] = 0.
            X_rec, _ = simulate_SL(T, dt, config['a'][0,0], config['omega'][0,0], config['beta'][0])
            print(f'Iteration {iter}', config)
            plt.plot(X_gt, label='GT')
            plt.plot(X_rec, label='Rec')
            plt.legend()
            plt.show()
         
        if iter == 0:
            factor_graph.update_all_observational_factors()
            
            for i in factor_graph.var_nodes:
                curr = factor_graph.var_nodes[i]
                curr.compute_and_send_messages()


        alpha_left, alpha_right = 0, 0,

        # Move the right pointer along
        for alpha_right in range(15, len(time), 15):
            var_nodes = [(t,r) for t in range(alpha_left, alpha_right) for r in range(nr)]
            for _ in range(10):
                random.shuffle(var_nodes)
                
                for t, r in var_nodes:
                    curr = factor_graph.var_nodes[f'X_t{t}']
                    curr.compute_and_send_messages()

                factor_nodes = [(t,r) for t in range(alpha_left, alpha_right-1) for r in range(nr)]
                random.shuffle(factor_nodes) 

                for t, r in factor_nodes:
                    factor_graph.factor_nodes[(f'osc_t{t}', f'osc_t{t+1}')].compute_and_send_messages()    

                factor_graph.update_params()

        # At the end, iterate backwards
        alpha_left, alpha_right = len(time)-1, len(time)-1
        for alpha_left in range(len(time), 0, -15):
            var_nodes = [(t,r) for t in range(alpha_left, alpha_right) for r in range(nr)]
            for _ in range(10):
                random.shuffle(var_nodes)

                for t, r in var_nodes:
                    curr = factor_graph.var_nodes[f'X_t{t}']
                    curr.compute_and_send_messages()

                factor_nodes = [(t,r) for t in range(alpha_left+1, alpha_right) for r in range(nr)]
                random.shuffle(factor_nodes) 

                for t, r in factor_nodes:
                    factor_graph.factor_nodes[(f'osc_t{t-1}', f'osc_t{t}')].compute_and_send_messages() 
                
                factor_graph.update_params()

        for j in factor_graph.param_ids: factor_graph.factor_nodes[f'{j}_prior'].belief = factor_graph.var_nodes[j].belief
        factor_graph.update_all_observational_factors()     
